# 🏛️ ggxx39 个人智慧文库 (Wisdom Library) · Colab 出版级高精重排工坊

> **遵循最高出版级精度标准**：Qwen2.5-VL-3B 多模态视觉大模型 + 物理裁切矩阵 + LaTeX 原生数学公式保真 + 单页断点续传 + 移动端流式 EPUB 3 与个人知识库 Markdown 双产物。
> **核心哲学**：知识和智慧之间，隔着一座大山。宁精勿滥，步步为营，每天精排 1~2 部经典名著，构建永恒智慧资产。

In [ ]:
# 步骤 1：GPU 算力锁定与出版级核心依赖安装 (Tesla T4 16GB)
import os, sys, subprocess, torch, gc

print("🔥 正在检测 GPU 算力...")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ 锁定 GPU: {gpu_name} (总显存: {vram_gb:.2f} GB)")
else:
    print("⚠️ 警告：当前未检测到 GPU！请在顶部菜单栏选择：运行时 -> 更改运行时类型 -> T4 GPU")

print("
📦 正在安装出版级解析依赖 (PyMuPDF, Transformers, Qwen-VL-Utils, Accelerate, Pandoc)...")
os.system("apt-get update -qq && apt-get install -y -qq pandoc")
os.system("pip install -q pymupdf transformers qwen-vl-utils accelerate torchvision")
print("✅ 依赖安装完毕！")

In [ ]:
# 步骤 2：生成 v6 iPhone Master 出版级排版样式表 (CSS)
css_code = """/* v6 iPhone Master Reader CSS - Grand Master Publication Edition */
@charset "UTF-8";
html, body {
    margin: 0; padding: 0 5%;
    font-family: -apple-system, "SF Pro Text", "PingFang SC", "Songti SC", "Source Han Serif SC", Georgia, serif;
    font-size: 1.05em; line-height: 1.82;
    color: #1c1c1e; background-color: #fbfbf9;
    text-align: justify; text-justify: inter-ideograph; word-break: break-word;
    -webkit-text-size-adjust: 100%;
}
h1 {
    font-size: 1.55em; line-height: 1.35; font-weight: 700; text-align: center;
    margin: 2.2em 0 1em; padding-bottom: 0.5em; border-bottom: 2.5px solid #007aff;
    page-break-before: always; color: #007aff;
}
h2 {
    font-size: 1.25em; line-height: 1.4; font-weight: 600;
    margin: 1.8em 0 0.8em; padding-bottom: 0.3em;
    border-bottom: 1.5px solid rgba(0, 122, 255, 0.25); color: #1c1c1e;
}
h3 {
    font-size: 1.08em; line-height: 1.45; font-weight: 600;
    margin: 1.3em 0 0.5em; padding-left: 0.5em;
    border-left: 3.5px solid #007aff; color: #2c3e50;
}
p { margin: 0.6em 0; text-indent: 0; }
strong, b { font-weight: 600; }
blockquote {
    margin: 1em 0; padding: 0.7em 1.1em;
    border-left: 3.5px solid #007aff; background: rgba(0, 122, 255, 0.05);
    border-radius: 0 6px 6px 0; font-size: 0.96em; color: #2c3e50;
}
table {
    width: 100% !important; max-width: 100% !important; border-collapse: separate; border-spacing: 0;
    margin: 1.2em 0; border: 1px solid rgba(128, 128, 128, 0.2); border-radius: 8px; overflow-x: auto; display: block;
}
th { background-color: rgba(0, 122, 255, 0.08); font-weight: 600; padding: 8px 12px; border-bottom: 1px solid rgba(128, 128, 128, 0.2); text-align: left; }
td { padding: 8px 12px; border-bottom: 1px solid rgba(128, 128, 128, 0.1); font-size: 0.95em; line-height: 1.6; }
@media (prefers-color-scheme: dark) {
    html, body { background-color: #121212; color: #e5e5ea; }
    h1 { color: #0a84ff; border-bottom-color: #0a84ff; }
    h2 { color: #f2f2f7; border-bottom-color: rgba(10, 132, 255, 0.3); }
    h3 { color: #0a84ff; border-left-color: #0a84ff; }
    blockquote { border-left-color: #0a84ff; background-color: rgba(10, 132, 255, 0.12); color: #e5e5ea; }
    th { background-color: rgba(10, 132, 255, 0.2); }
    td { border-bottom-color: rgba(255, 255, 255, 0.08); }
}
"""
with open("/content/iphone_master.css", "w", encoding="utf-8") as f:
    f.write(css_code)
print("✅ v6 iPhone Master CSS 已就绪: /content/iphone_master.css")

In [ ]:
# 步骤 3：秒级拉取 ggxx39/Books 线上藏书库 (免去手动上传 400MB)
print("📚 正在秒级同步 ggxx39/Books 线上藏书库...")
!git clone --depth 1 https://github.com/ggxx39/Books.git /content/Books_raw 2>/dev/null || (cd /content/Books_raw && git pull)
print("✅ 线上藏书库同步完毕！")

In [ ]:
# 步骤 4：查看书库并选定今日精排目标书目
from pathlib import Path

books_dir = Path("/content/Books_raw")
all_books = sorted([f.name for f in books_dir.iterdir() if f.suffix.lower() in [".pdf", ".epub"]])

print(f"📖 线上藏书库全量清单 (共 {len(all_books)} 部)：
")
for idx, b in enumerate(all_books, 1):
    print(f"  [{idx:02d}] {b}")

# 🎯 选定今日精排目标（默认首选数学启发式经典《怎样解题：数学思维的新方法》）：
TARGET_BOOK = "[怎样解题：数学思维的新方法].G·波利亚.pdf"
BOOK_TITLE = "怎样解题：数学思维的新方法"
CATEGORY = "02_数学与系统科学"

target_path = books_dir / TARGET_BOOK
print(f"
🚀 今日精排目标: {TARGET_BOOK}")
print(f"📦 归属分类: {CATEGORY}")

In [ ]:
# 步骤 5：启动最高精度出版级识别 (Qwen2.5-VL-3B + 物理裁切 + LaTeX + 断点续传)
import os, sys, re, time, io, subprocess, gc
import fitz
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

TARGET_PATH = str(target_path)
OUTPUT_BASE = Path("/content/精排电子书库_EPUB")
BOOK_OUT_DIR = OUTPUT_BASE / CATEGORY / BOOK_TITLE
BOOK_OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR = BOOK_OUT_DIR / ".checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

doc = fitz.open(TARGET_PATH)
total_pages = len(doc)
print(f"📄 全书总页数: {total_pages} 页")

# 1. 生成 2x Retina 封面
cover_path = BOOK_OUT_DIR / "cover.jpg"
pix = doc[0].get_pixmap(dpi=150)
pix.save(str(cover_path))
print(f"🖼️ Retina 封面已生成: {cover_path}")

# 2. 扫描已有断点缓存
cached_pages = {}
for cf in CHECKPOINTS_DIR.glob("page_*.md"):
    m = re.search(r"page_(\d+)\.md", cf.name)
    if m:
        cached_pages[int(m.group(1))] = cf.read_text(encoding="utf-8")
print(f"💾 已有断点页数: {len(cached_pages)} / {total_pages}")

# 3. 加载 Qwen2.5-VL-3B 多模态视觉模型
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
print(f"
🤖 正在加载视觉多模态大模型: {model_id} (float16, CUDA)...")
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="cuda"
)
print(f"✅ 模型加载完成！当前已分配显存: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

prompt = """你是一个世界顶级的数学科学出版物排版与OCR专家。请将这张书籍页面完整、精准、忠实地转录为出版级结构化 Markdown：
1. 【0漏字与公式保真】：数学符号、算式或定理必须严格转录为标准 LaTeX 格式（行内使用 $...$，独立公式使用 4356...4356）。绝对严禁丢漏符号。
2. 【标题识别】：根据排版字号和粗体精准识别章节标题或小节标题，使用相应的 Markdown 标题（#，## 等）。非标题的普通强调不要误用 #。
3. 【段落连贯】：段落之间保留一个空行。排版保持呼吸感和自然阅读流。
4. 【纯净输出】：直接输出转录后的结构化 Markdown，严禁输出任何多余的开场白或解释。"""

BATCH_SIZE = 4
for start_idx in range(0, total_pages, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_pages)
    batch_needed = [p for p in range(start_idx, end_idx) if p not in cached_pages]
    if not batch_needed:
        continue
    
    batch_msgs = []
    for pno in batch_needed:
        page = doc[pno]
        rect = page.rect
        # 物理裁切矩阵：顶端4.5%，底端5.5%（彻底消除页眉页脚）
        crop_box = fitz.Rect(0, rect.height * 0.045, rect.width, rect.height * 0.945)
        pix = page.get_pixmap(clip=crop_box, dpi=150)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        batch_msgs.append([
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img, "max_pixels": 400 * 28 * 28, "min_pixels": 256 * 28 * 28},
                    {"type": "text", "text": prompt}
                ]
            }
        ])
    
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_msgs]
    images_list = [process_vision_info(m)[0] for m in batch_msgs]
    inputs = processor(text=texts, images=images_list, padding=True, return_tensors="pt").to("cuda")
    
    t0 = time.time()
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
    elapsed = time.time() - t0
    
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    outputs = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    
    for pno, out_txt in zip(batch_needed, outputs):
        cached_pages[pno] = out_txt
        (CHECKPOINTS_DIR / f"page_{pno:04d}.md").write_text(out_txt, encoding="utf-8")
    
    print(f"  ⚡ 识别进度: {len(cached_pages)}/{total_pages} 页已完成 (本批耗时: {elapsed:.1f}s)")
    
    del inputs, generated_ids, trimmed, outputs
    gc.collect()
    torch.cuda.empty_cache()

doc.close()
print("
🎉 全书高精视觉推理识别全部完成！")

In [ ]:
# 步骤 6：双产物合成与 Pandoc 编译 (EPUB + Markdown with Frontmatter)
all_md = []
all_md.append("---")
all_md.append(f"title: "{BOOK_TITLE}"")
all_md.append("author: "G·波利亚"")
all_md.append(f"category: "{CATEGORY}"")
all_md.append(f"source_pdf: "{TARGET_BOOK}"")
all_md.append(f"processed_at: "{time.strftime('%Y-%m-%d')}"")
all_md.append("pipeline: "omni-pdf2epub (v6.0-qwen2.5-vl)"")
all_md.append("quality_rating: "Publication-Grade (A+)"")
all_md.append("tags: [数学思维, 启发式方法, 系统科学, 认知升级]")
all_md.append("---
")
all_md.append(f"# {BOOK_TITLE}
")

for pno in sorted(cached_pages.keys()):
    raw = cached_pages[pno]
    # 清除硬折行在中文间产生的孤立空格
    cleaned = re.sub(r"(?<=[一-龥])\s+(?=[一-龥])", "", raw)
    cleaned = re.sub(r"([一-龥])([a-zA-Z0-9])", r" ", cleaned)
    cleaned = re.sub(r"([a-zA-Z0-9])([一-龥])", r" ", cleaned)
    all_md.append(cleaned)
    all_md.append("
")

full_md_path = BOOK_OUT_DIR / f"{BOOK_TITLE}.md"
full_md_path.write_text("
".join(all_md), encoding="utf-8")
print(f"✅ 知识库 Markdown 生成完成: {full_md_path}")

epub_path = BOOK_OUT_DIR / f"{BOOK_TITLE}.epub"
print("📖 正在通过 Pandoc 编译出版级流式 EPUB 3...")
cmd = [
    "pandoc", str(full_md_path),
    "-o", str(epub_path),
    f"--resource-path={BOOK_OUT_DIR}",
    "--css=/content/iphone_master.css",
    f"--metadata=title:{BOOK_TITLE}",
    "--metadata=author:G·波利亚",
    "--metadata=language:zh-CN",
    "--toc",
    "--toc-depth=2",
    "--split-level=1",
    f"--epub-cover-image={cover_path}"
]
subprocess.run(cmd, check=True)
print(f"🎉 出版级流式 EPUB 编译成功: {epub_path} ({epub_path.stat().st_size / (1024*1024):.2f} MB)")

In [ ]:
# 步骤 7：一键打包双产物并触发浏览器自动下载
import shutil
from google.colab import files

ZIP_TARGET = "/content/wisdom_library_day1_assets"
shutil.make_archive(ZIP_TARGET, "zip", "/content/精排电子书库_EPUB")
zip_file = ZIP_TARGET + ".zip"
sz_mb = os.path.getsize(zip_file) / (1024 * 1024)
print(f"📦 双产物全量资产包已生成: {zip_file} ({sz_mb:.2f} MB)")
print("🚀 正在触发浏览器自动下载...")
files.download(zip_file)